# Backoff解析ノート

このノートブックは、PIMログから backoff 値ごとの**衝突率**と**合計スループット**と**1ノードあたりスループット** を解析し、PDFを出力します。

## 1. ライブラリの読み込み

解析と可視化に必要なライブラリを読み込みます。

In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd
import yaml

## 2. 入出力先設定

ログの入力先と、生成するPDFの保存先を設定します。必要に応じてこのセルだけ編集してください。

In [14]:
# ログルートとバックオフごとのフォルダー名を対応付ける
BACKOFF_ROOT_DIR = Path("../data/")
BACKOFF_FOLDER_MAP = {
    1024: "pim_20260501_054658_1024",
    2048: "pim_20260501_060930_2048",
    4096: "pim_20260501_063215_4096",
    8192: "pim_20260501_065454_8192",
    16384: "pim_20260501_071744_16384",
    32768: "pim_20260501_074032_32768",
    65536: "pim_20260501_080312_65536",
}

# ノード数の対象。必要に応じて編集する。
TARGET_NODES = [5, 10, 15, 20]

# PDF出力先
OUTPUT_DIR = Path("img/paper")
COLLISION_RATE_PDF = OUTPUT_DIR / "collision_rate_comparison_backoff_1024_65536.pdf"
THROUGHPUT_BPS_PDF = OUTPUT_DIR / "throughput_bps_comparison_backoff_1024_65536.pdf"
THROUGHPUT_PER_NODE_BPS_PDF = OUTPUT_DIR / "throughput_per_node_bps_comparison_backoff_1024_65536.pdf"

## 3. 解析コード

backoffごとの衝突率と **1ノードあたりスループット** を計算します。

In [15]:
def load_events(path: Path) -> pd.DataFrame:
    """events/evens TSVを読み込み、主要列を数値・文字列へ整形する。"""
    df = pd.read_csv(path, sep="\t")
    df["time_tu"] = pd.to_numeric(df["time_tu"], errors="coerce")
    df["round"] = pd.to_numeric(df["round"], errors="coerce").astype(int)
    df["event"] = df["event"].astype(str)
    df["result"] = df["result"].astype(str)
    return df


def load_config(config_path: Path) -> dict:
    """設定ファイルを読み込む。"""
    with config_path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def parse_detail_fields(detail: str) -> dict:
    """detail文字列を key=value 形式の辞書へ変換する。"""
    out = {}
    for item in str(detail).split(","):
        part = item.strip()
        if "=" not in part:
            continue
        key, value = part.split("=", 1)
        out[key.strip()] = value.strip()
    return out


def find_events_file(folder: Path, num_nodes: int) -> Path:
    """フォルダー内の evens/events TSV を返す。"""
    for candidate in (folder / f"evens_{num_nodes}.tsv", folder / f"events_{num_nodes}.tsv"):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"evens_{num_nodes}.tsv / events_{num_nodes}.tsv が見つかりません: {folder}")


def iter_backoff_event_files(
    root_dir: Path,
    backoff_folder_map: dict[int, str],
    target_nodes: list[int],
) -> list[tuple[int, int, Path]]:
    """バックオフ時間・ノード数・パスの組を返す。"""
    files = []
    for backoff, folder_name in backoff_folder_map.items():
        folder = root_dir / folder_name
        for nodes in target_nodes:
            events_path = find_events_file(folder, nodes)
            files.append((nodes, backoff, events_path))
    return sorted(files, key=lambda x: (x[0], x[1]))


def summarize_backoff(root_dir: Path, backoff_folder_map: dict[int, str], target_nodes: list[int]) -> pd.DataFrame:
    """backoffごとの衝突率と1ノードあたりスループット(bps)を集計して返す。"""
    event_files = iter_backoff_event_files(root_dir, backoff_folder_map, target_nodes)
    if not event_files:
        raise FileNotFoundError(
            f"evens_N.tsv / events_N.tsv が見つかりません: {root_dir}"
        )

    rows = []
    data_bits = None
    tu_sec = None
    for nodes, backoff, events_path in event_files:
        folder_config = events_path.parent / "config.yaml"
        if data_bits is None or tu_sec is None:
            config = load_config(folder_config)
            data_bits = float(config["data_bits"])
            tu_sec = float(config["tu"])
        elif folder_config.exists():
            config = load_config(folder_config)
            current_data_bits = float(config["data_bits"])
            current_tu_sec = float(config["tu"])
            if current_data_bits != data_bits or current_tu_sec != tu_sec:
                raise ValueError(f"config.yaml の内容がフォルダー間で異なります: {folder_config}")

        events = load_events(events_path)
        if events.empty:
            continue

        successes = int((events["event"] == "rx_success").sum())
        collision_rows = events[events["event"] == "collision"]

        collision_packets = 0
        for detail in collision_rows["detail"]:
            fields = parse_detail_fields(detail)
            try:
                collision_packets += int(float(fields.get("count", "1")))
            except ValueError:
                collision_packets += 1

        attempts = successes + collision_packets
        collision_rate = (collision_packets / attempts) if attempts > 0 else 0.0

        round_last_tu = events.groupby("round")["time_tu"].max()
        num_rounds = int(round_last_tu.shape[0])
        avg_round_tu = float(round_last_tu.mean()) if num_rounds > 0 else 0.0
        avg_round_sec = avg_round_tu * tu_sec
        avg_successes_per_round = (successes / num_rounds) if num_rounds > 0 else 0.0

        total_throughput_bps = (
            (avg_successes_per_round * data_bits / avg_round_sec)
            if avg_round_sec > 0
            else 0.0
        )
        throughput_per_node_bps = (total_throughput_bps / nodes) if nodes > 0 else 0.0

        rows.append(
            {
                "nodes": nodes,
                "backoff": backoff,
                "collisions": int(collision_rows.shape[0]),
                "successes": successes,
                "collision_rate": collision_rate,
                "total_throughput_bps": total_throughput_bps,
                "throughput_per_node_bps": throughput_per_node_bps,
                "file": events_path.name,
            }
        )

    return pd.DataFrame(rows).sort_values(["nodes", "backoff"]).reset_index(drop=True)


def plot_collision_rate(df: pd.DataFrame, output_pdf: Path, target_nodes: list[int]) -> None:
    """backoffに対する衝突率をPDFに出力する。"""
    fig, ax = plt.subplots(figsize=(8, 5))
    for node in target_nodes:
        subset = df[df["nodes"] == node].sort_values("backoff")
        if subset.empty:
            continue
        ax.plot(subset["backoff"], subset["collision_rate"], marker="o", label=f"nodes={node}")
    ax.set_title("Collision Rate vs Backoff")
    ax.set_xlabel("Backoff Max")
    ax.set_ylabel("Collision Rate")
    ax.set_xscale("log", base=2)
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    ax.legend()

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

def plot_throughputr_bps(df: pd.DataFrame, output_pdf: Path, target_nodes: list[int]) -> None:
    """backoffに対するスループット(bps)をPDFに出力する。"""
    fig, ax = plt.subplots(figsize=(8, 5))
    for node in target_nodes:
        subset = df[df["nodes"] == node].sort_values("backoff")
        if subset.empty:
            continue
        ax.plot(subset["backoff"], subset["total_throughput_bps"], marker="o", label=f"nodes={node}")
    ax.set_title("Total Throughput (bps) vs Backoff")
    ax.set_xlabel("Backoff Max")
    ax.set_ylabel("Total Throughput (bps)")
    ax.set_xscale("log", base=2)
    ax.grid(alpha=0.3)
    ax.legend()

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

def plot_throughput_per_node_bps(df: pd.DataFrame, output_pdf: Path, target_nodes: list[int]) -> None:
    """backoffに対する1ノードあたりスループット(bps)をPDFに出力する。"""
    fig, ax = plt.subplots(figsize=(8, 5))
    for node in target_nodes:
        subset = df[df["nodes"] == node].sort_values("backoff")
        if subset.empty:
            continue
        ax.plot(subset["backoff"], subset["throughput_per_node_bps"], marker="o", label=f"nodes={node}")
    ax.set_title("Throughput Per Node (bps) vs Backoff")
    ax.set_xlabel("Backoff Max")
    ax.set_ylabel("Throughput Per Node (bps)")
    ax.set_xscale("log", base=2)
    ax.grid(alpha=0.3)
    ax.legend()

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

## 4. 実行

このセルを実行すると、backoff解析を行ってPDFを出力し、結果テーブルを表示します。

In [16]:
result = summarize_backoff(BACKOFF_ROOT_DIR, BACKOFF_FOLDER_MAP, TARGET_NODES)

plot_collision_rate(result, COLLISION_RATE_PDF, target_nodes=TARGET_NODES)
plot_throughputr_bps(result, THROUGHPUT_BPS_PDF, target_nodes=TARGET_NODES)
plot_throughput_per_node_bps(result, THROUGHPUT_PER_NODE_BPS_PDF, target_nodes=TARGET_NODES)

print("backoff解析が完了しました。")
print(f"log root: {BACKOFF_ROOT_DIR}")
print(f"collision PDF: {COLLISION_RATE_PDF}")
print(f"throughput per node PDF: {THROUGHPUT_PER_NODE_BPS_PDF}")
print()
print(f"=== 解析結果 (nodes={TARGET_NODES}) ===")
display(result[["nodes", "backoff", "collision_rate", "throughput_per_node_bps"]])

backoff解析が完了しました。
log root: ../data
collision PDF: img/paper/collision_rate_comparison_backoff_1024_65536.pdf
throughput per node PDF: img/paper/throughput_per_node_bps_comparison_backoff_1024_65536.pdf

=== 解析結果 (nodes=[5, 10, 15, 20]) ===


,nodes,backoff,collision_rate,throughput_per_node_bps
0,5,1024,0.596570,1294.352502
1,5,2048,0.368298,1124.629601
2,5,4096,0.205366,746.701349
3,5,8192,0.108514,430.568840
4,5,16384,0.055582,231.261849
5,5,32768,0.027972,119.845540
6,5,65536,0.014406,60.972059
7,10,1024,0.861763,410.059675
8,10,2048,0.640803,590.113628
9,10,4096,0.402410,517.027628
